In [2]:
import duckdb
import pandas as pd

con = duckdb.connect()

# stations centrales sur chaque ligne
gares_a = [
    "Charles De Gaulle-Étoile",
    "Auber",
    "Châtelet-Les Halles",
    "Gare de Lyon",
    "Nation",
]

gares_b = [
    "Denfert-Rochereau",
    "Port Royal",
    "Luxembourg",
    "Saint-Michel Notre-Dame",
    "Châtelet-Les Halles",
    "Gare du Nord",
    "La Plaine Stade de France",
    "La Courneuve - Aubervilliers",
    "Le Bourget",
    "Drancy",
    "Le Blanc-Mesnil",
    "Aulnay-sous-Bois",
]

gares_d = [
    "Gare de Lyon",
    "Châtelet-Les Halles",
    "Gare du Nord",
    "Stade de France-Saint-Denis",
    "Saint-Denis",
]

In [ ]:
def init_s3():
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute("CREATE OR REPLACE SECRET (TYPE S3, KEY_ID '', SECRET '', ENDPOINT 'minio.lab.sspcloud.fr', URL_STYLE 'path');")


def charger_ligne(ligne, gares, source="local"):
    # DuckDB explose le JSON et filtre les gares en C++ (local ou s3)
    if source not in ("local", "s3"):
        raise ValueError("source doit valoir 'local' ou 's3'")

    if source == "local":
        chemin = f"../data/rer_{ligne}_2026-11-24.parquet"
        filtre_date = "WHERE date BETWEEN '2026-10-02' AND '2026-11-23'"
        params = [chemin, gares]
    else:
        init_s3()
        chemin = f"s3://gpichonlegroux/rer-{ligne}/date=*/*.parquet"
        filtre_date = "WHERE date >= '2026-10-02'"
        params = [chemin, gares]

    query = f"""
    WITH base AS (
        SELECT * EXCLUDE (stops),
               UNNEST(from_json(stops, '[{{"n":"VARCHAR","ad":"TIMESTAMPTZ","ed":"TIMESTAMPTZ","dt":"BIGINT","v":"VARCHAR","s":"VARCHAR"}}]')) AS stop
        FROM read_parquet(?, union_by_name=true)
        {filtre_date}
    ),
    ord AS (
        SELECT * EXCLUDE (stop),
               CASE WHEN stop.n = 'Châtelet - Les Halles' THEN 'Châtelet-Les Halles' ELSE stop.n END AS stop_name,
               stop.ad AS aimed_time,
               stop.ed AS expected_time,
               stop.dt AS delay_sec,
               stop.v AS platform,
               stop.s AS status,
               ROW_NUMBER() OVER (PARTITION BY date, journey_id ORDER BY stop.ad) AS stop_order,
               stop.dt - LAG(stop.dt, 1, 0) OVER (PARTITION BY date, journey_id ORDER BY stop.ad) AS delay_added_sec
        FROM base
    )
    SELECT * FROM ord
    WHERE list_contains(?, stop_name)
    ORDER BY date, journey_id, aimed_time
    """
    df = con.execute(query, params).df()
    df["line"] = df["line"].fillna(f"rer_{ligne}")
    df["aimed_time"] = df["aimed_time"].dt.tz_convert("Europe/Paris")
    df["expected_time"] = df["expected_time"].dt.tz_convert("Europe/Paris")
    return df


def charger_incidents(source="local"):
    if source not in ("local", "s3"):
        raise ValueError("source doit valoir 'local' ou 's3'")

    if source == "local":
        return con.execute(
            "SELECT * FROM read_parquet('../data/incidents_2026-11-24.parquet', union_by_name=true) WHERE date BETWEEN '2026-10-02' AND '2026-11-23'"
        ).df()

    init_s3()
    return con.execute(
        "SELECT * FROM read_parquet('s3://gpichonlegroux/incidents/date=*/*.parquet', union_by_name=true) WHERE date >= '2026-10-02'"
    ).df()


source = "local"  # passer à "s3" pour charger toutes les données du bucket public

df_rera = charger_ligne("a", gares_a, source=source)
df_rerb = charger_ligne("b", gares_b, source=source)
df_rerd = charger_ligne("d", gares_d, source=source)
df_incidents = charger_incidents(source=source)

In [ ]:
df_rera.columns

In [ ]:
# retard, suppressions, tunnel et créneaux horaires

def ajout_variables(df):
    df = df.copy()
    df["retard_min"] = df["delay_sec"] / 60
    df["est_supprime"] = df["status"] == "CANCELLED"
    df["zone_tunnel"] = df["stop_name"].isin(["Châtelet-Les Halles", "Gare du Nord"])

    semaine = df["aimed_time"].dt.day_of_week < 5
    heure = df["aimed_time"].dt.hour + df["aimed_time"].dt.minute / 60
    pointe_matin = semaine & (heure >= 7.5) & (heure < 9.5)
    pointe_soir = semaine & (heure >= 17.0) & (heure < 19.5)

    df["heure_pointe"] = pointe_matin | pointe_soir

    tranche = pd.Series("Heures creuses", index=df.index)
    tranche.loc[~semaine] = "Week-end"
    tranche.loc[semaine & ((heure >= 20.0) | (heure < 6.0))] = "Soirée"
    tranche.loc[pointe_matin] = "Pointe matin"
    tranche.loc[pointe_soir] = "Pointe soir"
    df["tranche_horaire"] = tranche
    return df

df_rera = ajout_variables(df_rera)
df_rerb = ajout_variables(df_rerb)
df_rerd = ajout_variables(df_rerd)

In [7]:
# controle et retards aberrants

for nom, df in [("RER A", df_rera), ("RER B", df_rerb), ("RER D", df_rerd)]:
    aberrants = (df["retard_min"].abs() > 180).sum()
    print(f"=== {nom} ({len(df)} arrêts) ===")
    print(f"  Suppressions : {df['est_supprime'].mean() * 100:.2f} % | Tunnel : {df['zone_tunnel'].mean() * 100:.1f} % | Pointe : {df['heure_pointe'].mean() * 100:.1f} %")
    print(f"  Retard (min) médian={df['retard_min'].median():.2f}, P90={df['retard_min'].quantile(0.9):.2f}, aberrants (|r|>180 min)={aberrants}")
    print("  Tranches :", df["tranche_horaire"].value_counts().to_dict())

=== RER A (148666 arrêts) ===
  Suppressions : 0.81 % | Tunnel : 20.2 % | Pointe : 27.2 %
  Retard (min) médian=0.37, P90=4.32, aberrants (|r|>180 min)=5
  Tranches : {'Heures creuses': 54935, 'Week-end': 32915, 'Pointe soir': 21442, 'Soirée': 20353, 'Pointe matin': 19021}
=== RER B (266064 arrêts) ===
  Suppressions : 2.00 % | Tunnel : 18.3 % | Pointe : 26.0 %
  Retard (min) médian=1.43, P90=8.47, aberrants (|r|>180 min)=1636
  Tranches : {'Heures creuses': 95537, 'Week-end': 61799, 'Soirée': 39571, 'Pointe soir': 37781, 'Pointe matin': 31376}
=== RER D (60314 arrêts) ===
  Suppressions : 4.21 % | Tunnel : 38.7 % | Pointe : 26.4 %
  Retard (min) médian=0.87, P90=7.75, aberrants (|r|>180 min)=1
  Tranches : {'Heures creuses': 25232, 'Week-end': 11104, 'Pointe soir': 9115, 'Soirée': 8074, 'Pointe matin': 6789}
